In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier

# ===== Config =====
SEED = 42
rng = np.random.default_rng(SEED)
N_SPLITS = 3
N_TRIALS_MAX = 10000  # hard cap; time budget usually stops earlier
MAX_HOURS = 10.0  # set 0.0 for one-trial dry run, set 9.0 for full search
USE_EARLY_STOPPING = True
EARLY_STOPPING_ROUNDS = 100
VERBOSE_EVERY = 100
SAVE_EVERY = 100

# ===== Data path auto-detect (local vs Kaggle) =====
local_train = '/workspaces/KagglePredictingStellarClass/data/train.csv'
kaggle_train = '/kaggle/input/competitions/playground-series-s6e6/train.csv'
train_path = local_train if os.path.exists(local_train) else kaggle_train
if not os.path.exists(train_path):
    raise FileNotFoundError('Could not find train.csv in local or Kaggle path.')

print('Train file:', train_path)
df = pd.read_csv(train_path)

# ===== Same preprocessing pattern as 02_baseline_rf.ipynb =====
df_eng = df.drop(columns=['id']).copy()
df_eng['u_g'] = df_eng['u'] - df_eng['g']
df_eng['g_r'] = df_eng['g'] - df_eng['r']
df_eng['r_i'] = df_eng['r'] - df_eng['i']
df_eng['i_z'] = df_eng['i'] - df_eng['z']
df_eng = df_eng.drop(columns=['u', 'g', 'r', 'i', 'z'])

for col in ['spectral_type', 'galaxy_population']:
    df_eng[col] = LabelEncoder().fit_transform(df_eng[col].astype(str))

X = df_eng.drop(columns=['class'])
y = LabelEncoder().fit_transform(df_eng['class'])
num_class = len(np.unique(y))

print('X shape:', X.shape)
print('Number of classes:', num_class)

# ===== Output setup =====
out_dir = '/kaggle/working/course_search_outputs' if os.path.exists('/kaggle/working') else '/workspaces/KagglePredictingStellarClass/course_search_outputs'
os.makedirs(out_dir, exist_ok=True)
csv_path = os.path.join(out_dir, 'xgb_course_random_search_results.csv')
json_path = os.path.join(out_dir, 'xgb_course_random_search_best.json')

def sample_params(rng_local):
    return {
        # Wide tree cap is intentional when early stopping is enabled.
        'n_estimators': int(rng_local.integers(1000, 4000)),
        'learning_rate': float(rng_local.uniform(0.01, 0.15)),
        'max_depth': int(rng_local.integers(3, 11)),
        'min_child_weight': float(rng_local.uniform(1, 12)),
        'subsample': float(rng_local.uniform(0.6, 1.0)),
        'colsample_bytree': float(rng_local.uniform(0.6, 1.0)),
        'reg_alpha': float(10 ** rng_local.uniform(-4, 1)),
        'reg_lambda': float(10 ** rng_local.uniform(-3, 2)),
        'gamma': float(rng_local.uniform(0, 5)),
        'n_jobs': -1
    }

def save_results(results_list):
    if not results_list:
        return
    tmp_df = pd.DataFrame(results_list).sort_values('mean_balanced_accuracy', ascending=False).reset_index(drop=True)
    tmp_df.to_csv(csv_path, index=False)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(tmp_df.iloc[0].to_dict(), f, indent=2)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
results = []
overall_start = time.perf_counter()
max_seconds = MAX_HOURS * 3600.0

if MAX_HOURS <= 0:
    print('Dry run mode: MAX_HOURS <= 0, will run exactly 1 trial.')
else:
    print(f'Time budget: {MAX_HOURS:.2f} hours')

trial = 1
while trial <= N_TRIALS_MAX:
    elapsed_before_trial = time.perf_counter() - overall_start
    if MAX_HOURS > 0 and elapsed_before_trial >= max_seconds:
        print('Time budget reached before starting next trial. Stopping.')
        break

    params = sample_params(rng)
    fold_scores = []
    fit_times = []
    pred_times = []
    trial_start = time.perf_counter()

    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr = X.iloc[tr_idx]
        y_tr = y[tr_idx]
        X_va = X.iloc[va_idx]
        y_va = y[va_idx]

        common = dict(
            objective='multi:softprob',
            num_class=num_class,
            eval_metric='mlogloss',
            tree_method='hist',
            random_state=SEED
        )
        if USE_EARLY_STOPPING:
            common['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS

        fit_start = time.perf_counter()
        try:
            model = XGBClassifier(device='cuda', **params, **common)
            if USE_EARLY_STOPPING:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            else:
                model.fit(X_tr, y_tr, verbose=False)
        except Exception:
            model = XGBClassifier(device='cpu', **params, **common)
            if USE_EARLY_STOPPING:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            else:
                model.fit(X_tr, y_tr, verbose=False)
        fit_elapsed = time.perf_counter() - fit_start

        pred_start = time.perf_counter()
        pred = model.predict(X_va)
        pred_elapsed = time.perf_counter() - pred_start

        score = balanced_accuracy_score(y_va, pred)
        fold_scores.append(float(score))
        fit_times.append(float(fit_elapsed))
        pred_times.append(float(pred_elapsed))

    trial_elapsed = time.perf_counter() - trial_start
    row = {
        'trial': trial,
        'search_seed': SEED,
        'use_early_stopping': USE_EARLY_STOPPING,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS if USE_EARLY_STOPPING else 0,
        'mean_balanced_accuracy': float(np.mean(fold_scores)),
        'std_balanced_accuracy': float(np.std(fold_scores)),
        'fold_scores': fold_scores,
        'fit_time_sec_mean': float(np.mean(fit_times)),
        'pred_time_sec_mean': float(np.mean(pred_times)),
        'trial_time_sec': float(trial_elapsed),
        **params
    }
    results.append(row)

    elapsed_after_trial = time.perf_counter() - overall_start
    if trial == 1 or trial % VERBOSE_EVERY == 0:
        if MAX_HOURS > 0:
            remaining = max(0.0, max_seconds - elapsed_after_trial)
            print(
                f'Trial {trial} | mean_bal_acc={row["mean_balanced_accuracy"]:.6f} | '
                f'trial_time={trial_elapsed:.2f}s | elapsed={(elapsed_after_trial/3600):.2f}h | '
                f'remaining={(remaining/3600):.2f}h'
            )
        else:
            print(
                f'Trial {trial} | mean_bal_acc={row["mean_balanced_accuracy"]:.6f} | '
                f'trial_time={trial_elapsed:.2f}s'
            )

    if trial % SAVE_EVERY == 0 or trial == 1:
        save_results(results)

    if MAX_HOURS <= 0:
        print('Dry run complete (1 trial).')
        break

    if elapsed_after_trial >= max_seconds:
        print('Time budget reached after trial completion. Stopping.')
        break

    trial += 1

# Final save and summary
save_results(results)

if results:
    results_df = pd.DataFrame(results).sort_values('mean_balanced_accuracy', ascending=False).reset_index(drop=True)
    elapsed_total = time.perf_counter() - overall_start

    print('\n=== Search Complete ===')
    print(f'Trials completed: {len(results)}')
    print(f'Total search time (minutes): {elapsed_total / 60:.2f}')
    print('Top 10 trials by balanced accuracy:')
    display(results_df.head(10))
    print('Saved full results CSV:', csv_path)
    print('Saved best trial JSON:', json_path)
else:
    print('No trials were completed. Nothing saved.')